# 알약 검수 파이프라인 서버 (Colab)

**사전 준비 (Google Drive에 업로드):**
```
MyDrive/pill_project/
  inspection_pipeline/   ← pipeline.py, server.py 등 파이프라인 파일들
  model/
    rtmdet_single_pill_aug_300.py
    best_coco_bbox_mAP_50_epoch_273.pth
```

**런타임 → 런타임 유형 변경 → GPU (T4) 선택 필수**

In [1]:
import os
for v in ['3.10', '3.11', '3.9']:
    path = f'/usr/bin/python{v}'
    if os.path.exists(path):
        print(f"사용 가능: {path}")

사용 가능: /usr/bin/python3.10


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# [1] GPU 확인
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '없음 - 런타임을 GPU로 변경하세요')

GPU: Tesla T4


In [4]:
import sys
!{sys.executable} -m pip install -U pip setuptools wheel
print(f"pip path: {sys.executable}")

pip path: /usr/bin/python3


In [5]:
import sys
!{sys.executable} -m pip --version

pip 26.1.1 from /usr/local/lib/python3.12/dist-packages/pip (python 3.12)


In [6]:
import sys
!{sys.executable} -m pip uninstall -y torchaudio || true
!{sys.executable} -m pip install -q torch==2.3.0 torchvision==0.18.0 --index-url https://download.pytorch.org/whl/cu121
!{sys.executable} -m pip install -q mmengine
!{sys.executable} -m pip install -q mmcv -f https://download.openmmlab.com/mmcv/dist/cu121/torch2.3/index.html
!{sys.executable} -m pip install -q mmdet
!{sys.executable} -m pip install -q fastapi uvicorn python-multipart easyocr scipy
!{sys.executable} -m pip install -q pyngrok
print("설치 완료")

설치 완료


In [7]:
# import sys
# !{sys.executable} -m pip install torch==2.3.0 torchvision==0.18.0 --index-url https://download.pytorch.org/whl/cu121 -q
# !{sys.executable} -m pip install mmcv -f https://download.openmmlab.com/mmcv/dist/cu121/torch2.3/index.html -q
# print("완료")

In [8]:
# [3] Google Drive 마운트 및 파일 복사
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
src = '/content/drive/MyDrive/pill_project'
dst = '/content/pill_project'
if os.path.exists(dst):
    shutil.rmtree(dst)
shutil.copytree(src, dst)
print('복사 완료:')
!ls /content/pill_project/inspection_pipeline/

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
복사 완료:
ocr_reader.py	  pipeline.py	     server.py
pill_detector.py  pouch_splitter.py  visualizer.py


In [9]:
# [4] 모델 경로 환경변수 설정
import os
os.environ['PILL_CONFIG']     = '/content/pill_project/model/rtmdet_single_pill_aug_300.py'
os.environ['PILL_CHECKPOINT'] = '/content/pill_project/model/best_coco_bbox_mAP_50_epoch_273.pth'
print('설정 완료')

설정 완료


In [10]:
init_path = '/usr/local/lib/python3.12/dist-packages/mmdet/__init__.py'
with open(init_path, 'r') as f:
    c = f.read()
c = c.replace("mmcv_maximum_version = '2.2.0'", "mmcv_maximum_version = '2.3.0'")
with open(init_path, 'w') as f:
    f.write(c)
print("패치 완료")

패치 완료


In [11]:
# This cell was a duplicate for pyngrok installation and has been consolidated.

In [12]:
import subprocess, threading, time, os, sys
from pyngrok import ngrok

NGROK_TOKEN = '3DZ57MhkCpvpLDb5ddsTEaeFRe2_5cAG4pKWJ5LhgFwrjsNap'
ngrok.set_auth_token(NGROK_TOKEN)

env = os.environ.copy()
env.pop('MPLBACKEND', None)

log_file = open('/tmp/server.log', 'w')

def run_server():
    subprocess.run(
        [sys.executable, '-m', 'uvicorn', 'server:app',
         '--host', '0.0.0.0', '--port', '8000'],
        cwd='/content/pill_project/inspection_pipeline',
        stdout=log_file, stderr=log_file,
        env=env
    )

threading.Thread(target=run_server, daemon=True).start()
time.sleep(8)
tunnel = ngrok.connect(8000)
SERVER_URL = tunnel.public_url
print(f'서버 주소: {SERVER_URL}')
print(f'분석 엔드포인트: {SERVER_URL}/analyze')

서버 주소: https://knapsack-sway-automaker.ngrok-free.dev
분석 엔드포인트: https://knapsack-sway-automaker.ngrok-free.dev/analyze


In [13]:
import sys
print(sys.executable)
print(sys.version)

/usr/bin/python3
3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
